In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
open System
open System.Net.Http
open System.Text.RegularExpressions

# A Simple Email Harvester

We use .NET `Regex` for patterns, `HttpClient` for optional HTTP retrieval, and `Uri` to resolve relative links and compare hosts. The traversal accepts a page retrieval function so the default example is deterministic and offline.

First, we compile some regular expressions into finite state machines and stores these FSMs
in global variables.
1. The regular expression `r'\<span style="display: none;"\>[^<>]*\</span\>'`
   is used because most email adresses are disguised as follows:
   ```
   karl.stroetmann<span style="display: none;"> No Spam \</span>@dhbw-mannheim.de`
   ```
   We have to remove the part `<span style="display: none;"> No Spam \</span>` from the web page
   so that email adresses can be recognized.
2. The regular expression `r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'` is used
   to locate email addresses.
3. The regular expression `r'<a [^>]*href=["\'](.*?)["\']'` is used
   to locate hypertext links.

In [ ]:
let invisible = Regex(@"<span style=""display: none;"">[^<>]*</span>")
let email = Regex(@"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}")
let link = Regex(@"<a [^>]*href=[""'](.*?)[""']")

The function `retrieve_page(url)` retrieves the text of the web page at the given `url`.
It also removes text like `<span style="display: none;"\> No Spam \</span>` from this text.
Text of this kind is sometimes inserted in email adresses to make it more difficult to
harvest them.

In [ ]:
let removeInvisible page = invisible.Replace(page, "")
// Optional real HTTP boundary. Calling it is explicit; the notebook does not crawl the Internet.
let retrievePage (client: HttpClient) (url: Uri) = async {
    let! page = client.GetStringAsync(url) |> Async.AwaitTask
    return removeInvisible page
}

`extractEmails retrieve start maximumPages` traverses reachable pages on the original host, ignores fragments and external hosts, and visits each page at most once. Retrieval is injected; an explicit page budget bounds the traversal. The default fixture is offline.

In [ ]:
let extractEmails (retrieve: Uri -> string) (start: Uri) maximumPages =
    if maximumPages <= 0 then invalidArg "maximumPages" "Use a positive page budget"
    let rec visit (pending: Uri list) processed emails =
        match pending with
        | [] -> emails
        | _ when Set.count processed >= maximumPages -> emails
        | url::rest when Set.contains url.AbsoluteUri processed -> visit rest processed emails
        | url::rest ->
            let page = retrieve url |> removeInvisible
            let found = email.Matches(page) |> Seq.cast<Match> |> Seq.map (fun m -> m.Value) |> Set.ofSeq
            let targets =
                link.Matches(page) |> Seq.cast<Match>
                |> Seq.choose (fun m ->
                    match Uri.TryCreate(url, m.Groups.[1].Value) with
                    | true, target when target.Host = start.Host && (target.Scheme = "http" || target.Scheme = "https") ->
                        let canonical = UriBuilder(target)
                        canonical.Fragment <- ""
                        Some canonical.Uri
                    | _ -> None)
                |> Seq.toList
            visit (targets @ rest) (Set.add url.AbsoluteUri processed) (Set.union emails found)
    visit [start] Set.empty Set.empty

In [ ]:
let pages = Map.ofList [
    "https://lecture.example/", """<a href="/about">About</a> teacher<span style="display: none;"> No Spam </span>@lecture.example"""
    "https://lecture.example/about", """assistant@lecture.example <a href="/#top">Home</a><a href="https://other.example/">External</a>""" ]
let emails = extractEmails (fun url -> pages.[url.AbsoluteUri]) (Uri("https://lecture.example/")) 10
emails |> Set.iter (printfn "%s")
checkNotebook (emails = Set.ofList ["teacher@lecture.example"; "assistant@lecture.example"])